In [ ]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from typing import TypedDict, Annotated
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, BaseMessage
from langgraph.graph.message import add_messages


In [ ]:
load_dotenv()

In [ ]:
model = ChatGroq(model="openai/gpt-oss-20b")

In [ ]:
class ChatState(TypedDict):
    
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
graph = StateGraph(ChatState)

In [ ]:
def chat_node(state: ChatState):
    
    messages = state['messages']
    
    response = model.invoke(messages)

    return {
        "messages" : [response]
    }

In [ ]:
graph.add_node("chat_node", chat_node)

graph.add_edge(START, "chat_node")
graph.add_edge("chat_node", END)

workflow = graph.compile()
workflow

In [ ]:
while True:
    
    user_input = input("Type here:")
    print(user_input)
    
    if user_input.strip().lower() in ["exit", "quit", "bye"]:
        break

    response = workflow.invoke({"messages": [HumanMessage(content=user_input)]})
    print("AI:", response['messages'][-1].content)

